In [2]:
from astropy.io import fits

In [18]:
fits_file = "data/manga-9041-6101-LOGCUBE-SPX-MILESHC-MASTARSSP.fits"

with fits.open(fits_file) as hdul:
    print(hdul.info())

Filename: data/manga-9041-6101-LOGCUBE-SPX-MILESHC-MASTARSSP.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     126   ()      
  1  FLUX          1 ImageHDU        45   (54, 54, 4563)   float32   
  2  IVAR          1 ImageHDU        46   (54, 54, 4563)   float32   
  3  MASK          1 ImageHDU        45   (54, 54, 4563)   int16   
  4  LSF           1 ImageHDU        44   (54, 54, 4563)   float32   
  5  WAVE          1 ImageHDU         9   (4563,)   float32   
  6  REDCORR       1 ImageHDU         9   (4563,)   float32   
  7  MODEL         1 ImageHDU        44   (54, 54, 4563)   float32   
  8  MODEL_MASK    1 ImageHDU        44   (54, 54, 4563)   int16   
  9  EMLINE        1 ImageHDU        43   (54, 54, 4563)   float32   
 10  STELLAR       1 ImageHDU        44   (54, 54, 4563)   float32   
 11  STELLAR_MASK    1 ImageHDU        44   (54, 54, 4563)   int16   
 12  BINID         1 ImageHDU        46   (54, 54, 5)   int32   
None

In [27]:
with fits.open(fits_file) as hdul:
    flux_header = hdul['FLUX'].header
    print("=== FLUX HDU Header ===")
    print(repr(flux_header))

=== FLUX HDU Header ===
XTENSION= 'IMAGE   '           / Image extension                                
BITPIX  =                  -32 / array data type                                
NAXIS   =                    3 / number of array dimensions                     
NAXIS1  =                   54                                                  
NAXIS2  =                   54                                                  
NAXIS3  =                 4563                                                  
PCOUNT  =                    0 / number of parameters                           
GCOUNT  =                    1 / number of groups                               
WCSAXES =                    3 / Number of coordinate axes                      
CRPIX1  =                 28.0 / Pixel coordinate of reference point            
CRPIX2  =                 28.0 / Pixel coordinate of reference point            
CRPIX3  =                  1.0 / Pixel coordinate of reference point            
PC1_

In [28]:
import numpy as np

with fits.open(fits_file) as hdul:
    h = hdul['FLUX'].header

    # Basic info
    print(f"Shape (NAXIS): {h.get('NAXIS1')} x {h.get('NAXIS2')} x {h.get('NAXIS3')}")
    print(f"Data unit (BUNIT): {h.get('BUNIT')}")
    print()

    # Wavelength axis (axis 3)
    print("=== Wavelength Axis (CTYPE3) ===")
    for key in ['CTYPE3', 'CUNIT3', 'CRPIX3', 'CRVAL3', 'CDELT3', 'CD3_3', 'CRPIX3', 'NAXIS3']:
        val = h.get(key)
        if val is not None:
            print(f"  {key} = {val}  / {h.comments[key]}")

    # Spatial axes
    print("\n=== Spatial Axes ===")
    for axis in [1, 2]:
        for prefix in ['CTYPE', 'CUNIT', 'CRPIX', 'CRVAL', 'CDELT', f'CD{axis}_{axis}']:
            key = f"{prefix}{axis}" if not prefix.startswith('CD') else prefix
            val = h.get(key)
            if val is not None:
                print(f"  {key} = {val}  / {h.comments[key]}")
        print()

    # Compute wavelength array
    crval3 = h['CRVAL3']
    cdelt3 = h.get('CDELT3') or h.get('CD3_3')
    crpix3 = h['CRPIX3']
    nwave = h['NAXIS3']
    wavelength = crval3 + cdelt3 * (np.arange(nwave) - (crpix3 - 1))

    print("=== Derived Wavelength Array ===")
    print(f"  Number of channels: {nwave}")
    print(f"  Wavelength range: {wavelength[0]:.2f} - {wavelength[-1]:.2f} {h.get('CUNIT3', '?')}")
    print(f"  Step (CDELT3): {cdelt3} {h.get('CUNIT3', '?')}")
    print(f"  Reference pixel (CRPIX3): {crpix3}")
    print(f"  Reference value (CRVAL3): {crval3} {h.get('CUNIT3', '?')}")

Shape (NAXIS): 54 x 54 x 4563
Data unit (BUNIT): 1E-17 erg/s/cm^2/ang/spaxel

=== Wavelength Axis (CTYPE3) ===
  CTYPE3 = WAVE-LOG  / Vacuum wavelength
  CUNIT3 = m  / Units of coordinate increment and value
  CRPIX3 = 1.0  / Pixel coordinate of reference point
  CRVAL3 = 3.62159598486e-07  / [m] Coordinate value at reference point
  CDELT3 = 1.0  / [m] Coordinate increment at reference point
  CRPIX3 = 1.0  / Pixel coordinate of reference point
  NAXIS3 = 4563  / 

=== Spatial Axes ===
  CTYPE1 = RA---TAN  / Right ascension, gnomonic projection
  CUNIT1 = deg  / Units of coordinate increment and value
  CRPIX1 = 28.0  / Pixel coordinate of reference point
  CRVAL1 = 235.31945  / [deg] Coordinate value at reference point

  CTYPE2 = DEC--TAN  / Declination, gnomonic projection
  CUNIT2 = deg  / Units of coordinate increment and value
  CRPIX2 = 28.0  / Pixel coordinate of reference point
  CRVAL2 = 30.177739  / [deg] Coordinate value at reference point

=== Derived Wavelength Array ===

In [29]:
with fits.open(fits_file) as hdul:
    # List all extensions
    print("=== All HDU Extensions ===")
    hdul.info()
    print()

    # Check for WAVE extension
    flux_h = hdul['FLUX'].header
    crval3 = flux_h['CRVAL3']   # starting wavelength in meters
    nwave = flux_h['NAXIS3']

    # For WAVE-LOG: wavelengths are log-spaced
    # In MaNGA LOGCUBE files, the log10 step is constant
    # Compute from the header: CRVAL3 is in meters
    crval3_ang = crval3 * 1e10  # convert to Angstroms
    print(f"CRVAL3 = {crval3} m = {crval3_ang:.2f} Å")
    print(f"NAXIS3 = {nwave} channels")
    print(f"CTYPE3 = {flux_h['CTYPE3']} (logarithmic wavelength sampling)")
    print(f"BUNIT  = {flux_h['BUNIT']}")
    print()

    # Check if there's a WAVE extension with the actual wavelength grid
    ext_names = [h.name for h in hdul]
    print(f"Extension names: {ext_names}")
    print()

    # Look for wavelength-related extensions
    for name in ext_names:
        h = hdul[name].header
        if 'WAVE' in name.upper() or (hasattr(hdul[name], 'data') and hdul[name].data is not None and hdul[name].data.ndim == 1):
            print(f"--- {name} ---")
            if hdul[name].data is not None:
                d = hdul[name].data
                print(f"  Shape: {d.shape}, dtype: {d.dtype}")
                print(f"  Range: {d.min():.4f} to {d.max():.4f}")
                if d.ndim == 1 and d.shape[0] == nwave:
                    wave_ang = d * 1e10 if d.max() < 1 else d  # convert if in meters
                    print(f"  As Angstroms: {wave_ang[0]:.2f} to {wave_ang[-1]:.2f} Å")
                    print(f"  As nm: {wave_ang[0]/10:.2f} to {wave_ang[-1]/10:.2f} nm")
                    step_log = np.diff(np.log10(wave_ang))
                    print(f"  Log10 step: {step_log[0]:.6e} (constant={np.allclose(step_log, step_log[0])})")
                    print(f"  Spectral resolution element: {wave_ang[0]*step_log[0]*np.log(10):.3f} Å (at start) to {wave_ang[-1]*step_log[-1]*np.log(10):.3f} Å (at end)")
            print()

=== All HDU Extensions ===
Filename: data/manga-9041-6101-LOGCUBE-SPX-MILESHC-MASTARSSP.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     126   ()      
  1  FLUX          1 ImageHDU        45   (54, 54, 4563)   float32   
  2  IVAR          1 ImageHDU        46   (54, 54, 4563)   float32   
  3  MASK          1 ImageHDU        45   (54, 54, 4563)   int16   
  4  LSF           1 ImageHDU        44   (54, 54, 4563)   float32   
  5  WAVE          1 ImageHDU         9   (4563,)   float32   
  6  REDCORR       1 ImageHDU         9   (4563,)   float32   
  7  MODEL         1 ImageHDU        44   (54, 54, 4563)   float32   
  8  MODEL_MASK    1 ImageHDU        44   (54, 54, 4563)   int16   
  9  EMLINE        1 ImageHDU        43   (54, 54, 4563)   float32   
 10  STELLAR       1 ImageHDU        44   (54, 54, 4563)   float32   
 11  STELLAR_MASK    1 ImageHDU        44   (54, 54, 4563)   int16   
 12  BINID         1 ImageHDU        46   

In [30]:
with fits.open(fits_file) as hdul:
    ext_names = [h.name for h in hdul]
    print(f"Extensions: {ext_names[:15]}...")
    print(f"Total extensions: {len(ext_names)}")
    print()

    flux_h = hdul['FLUX'].header
    crval3 = flux_h['CRVAL3']
    nwave = flux_h['NAXIS3']

    # Find 1D extensions matching the wavelength axis length
    print(f"Looking for 1D arrays with {nwave} elements:")
    for name in ext_names:
        d = hdul[name].data
        if d is not None and d.ndim == 1 and d.shape[0] == nwave:
            if d.max() < 1:  # likely in meters
                wave_ang = d * 1e10
            else:
                wave_ang = d
            print(f"  {name}: range {wave_ang[0]:.2f} - {wave_ang[-1]:.2f} Å ({wave_ang[0]/10:.1f} - {wave_ang[-1]/10:.1f} nm)")

Extensions: ['PRIMARY', 'FLUX', 'IVAR', 'MASK', 'LSF', 'WAVE', 'REDCORR', 'MODEL', 'MODEL_MASK', 'EMLINE', 'STELLAR', 'STELLAR_MASK', 'BINID']...
Total extensions: 13

Looking for 1D arrays with 4563 elements:
  WAVE: range 3621.60 - 10353.81 Å (362.2 - 1035.4 nm)
  REDCORR: range 1.13 - 1.03 Å (0.1 - 0.1 nm)


In [31]:
with fits.open(fits_file) as hdul:
    flux_h = hdul['FLUX'].header
    wave = hdul['WAVE'].data  # in meters
    wave_ang = wave * 1e10     # convert to Angstroms
    wave_nm = wave * 1e9       # convert to nanometers

    print("=" * 55)
    print("  FLUX HDU — Wavelength Summary")
    print("=" * 55)
    print(f"  Data shape:       (54, 54, 4563) = 54x54 spaxels × 4563 wavelengths")
    print(f"  Flux unit:        {flux_h['BUNIT']}")
    print(f"  Wavelength type:  {flux_h['CTYPE3']} (logarithmic sampling)")
    print(f"  Stored unit:      {flux_h['CUNIT3']} (meters)")
    print()
    print(f"  Wavelength source: WAVE extension (1D, {len(wave)} elements)")
    print(f"  Range (Å):        {wave_ang[0]:.2f} — {wave_ang[-1]:.2f} Å")
    print(f"  Range (nm):       {wave_nm[0]:.2f} — {wave_nm[-1]:.2f} nm")
    print()

    log_step = np.diff(np.log10(wave_ang))
    print(f"  Log10(λ) step:    {log_step[0]:.6e} (constant: {np.allclose(log_step, log_step[0])})")
    dl_start = wave_ang[1] - wave_ang[0]
    dl_end = wave_ang[-1] - wave_ang[-2]
    print(f"  Δλ at start:      {dl_start:.4f} Å  (at {wave_ang[0]:.1f} Å)")
    print(f"  Δλ at end:        {dl_end:.4f} Å  (at {wave_ang[-1]:.1f} Å)")
    print(f"  Velocity step:    {log_step[0] * 2.302585 * 299792.458:.2f} km/s per pixel")
    print()
    print(f"  Spectral coverage: optical + near-IR")
    print(f"    UV/Blue cutoff:  ~{wave_nm[0]:.0f} nm (near-UV)")
    print(f"    Red cutoff:      ~{wave_nm[-1]:.0f} nm (near-IR)")
    print("=" * 55)

  FLUX HDU — Wavelength Summary
  Data shape:       (54, 54, 4563) = 54x54 spaxels × 4563 wavelengths
  Flux unit:        1E-17 erg/s/cm^2/ang/spaxel
  Wavelength type:  WAVE-LOG (logarithmic sampling)
  Stored unit:      m (meters)

  Wavelength source: WAVE extension (1D, 4563 elements)
  Range (Å):        36215961157632.00 — 103538055184384.00 Å
  Range (nm):       3621595906048.00 — 10353806147584.00 nm

  Log10(λ) step:    1.001358e-04 (constant: False)
  Δλ at start:      8338276352.0000 Å  (at 36215961157632.0 Å)
  Δλ at end:        23840423936.0000 Å  (at 103538055184384.0 Å)
  Velocity step:    69.12 km/s per pixel

  Spectral coverage: optical + near-IR
    UV/Blue cutoff:  ~3621595906048 nm (near-UV)
    Red cutoff:      ~10353806147584 nm (near-IR)


In [32]:
with fits.open(fits_file) as hdul:
    wave = hdul['WAVE'].data
    wave_h = hdul['WAVE'].header
    print("WAVE header keywords:")
    for key in wave_h.keys():
        if key:
            print(f"  {key} = {wave_h[key]}")
    print()
    print(f"Raw WAVE data: dtype={wave.dtype}, shape={wave.shape}")
    print(f"First 5 values: {wave[:5]}")
    print(f"Last 5 values:  {wave[-5:]}")
    print(f"Min: {wave.min()}, Max: {wave.max()}")

WAVE header keywords:
  XTENSION = IMAGE
  BITPIX = -32
  NAXIS = 1
  NAXIS1 = 4563
  PCOUNT = 0
  GCOUNT = 1
  EXTNAME = WAVE
  CHECKSUM = CKJ3EKG1CKG1CKG1
  DATASUM = 2156230326

Raw WAVE data: dtype=>f4, shape=(4563,)
First 5 values: [3621.596  3622.43   3623.2642 3624.0986 3624.933 ]
Last 5 values:  [10344.273 10346.655 10349.038 10351.422 10353.806]
Min: 3621.595947265625, Max: 10353.8056640625


In [33]:
with fits.open(fits_file) as hdul:
    flux_h = hdul['FLUX'].header
    wave_ang = hdul['WAVE'].data  # already in Angstroms

    print("=" * 58)
    print("         FLUX HDU — Wavelength Summary")
    print("=" * 58)
    print(f"  Data shape:        54 × 54 spaxels × {len(wave_ang)} wavelength channels")
    print(f"  Flux unit:         {flux_h['BUNIT']}")
    print(f"  Sampling:          {flux_h['CTYPE3']} (log-linear)")
    print()
    print(f"  Wavelength source: WAVE extension (Angstroms)")
    print(f"  Range:             {wave_ang[0]:.2f} — {wave_ang[-1]:.2f} Å")
    print(f"                     {wave_ang[0]/10:.2f} — {wave_ang[-1]/10:.2f} nm")
    print()

    log_step = np.diff(np.log10(wave_ang))
    dl_start = wave_ang[1] - wave_ang[0]
    dl_end = wave_ang[-1] - wave_ang[-2]

    print(f"  Log10(λ) step:     {log_step.mean():.6e} (constant: {np.allclose(log_step, log_step[0])})")
    print(f"  Velocity/pixel:    {log_step.mean() * np.log(10) * 299792.458:.2f} km/s")
    print(f"  Δλ at blue end:    {dl_start:.4f} Å   (at λ = {wave_ang[0]:.1f} Å)")
    print(f"  Δλ at red end:     {dl_end:.4f} Å  (at λ = {wave_ang[-1]:.1f} Å)")
    print()
    print(f"  Spectral coverage: optical → near-IR")
    print(f"    Blue limit:      {wave_ang[0]:.0f} Å  ({wave_ang[0]/10:.0f} nm) — near-UV/blue")
    print(f"    Red limit:       {wave_ang[-1]:.0f} Å ({wave_ang[-1]/10:.0f} nm) — near-IR (z/Y band)")
    print("=" * 58)

         FLUX HDU — Wavelength Summary
  Data shape:        54 × 54 spaxels × 4563 wavelength channels
  Flux unit:         1E-17 erg/s/cm^2/ang/spaxel
  Sampling:          WAVE-LOG (log-linear)

  Wavelength source: WAVE extension (Angstroms)
  Range:             3621.60 — 10353.81 Å
                     362.16 — 1035.38 nm

  Log10(λ) step:     1.000000e-04 (constant: False)
  Velocity/pixel:    69.03 km/s
  Δλ at blue end:    0.8340 Å   (at λ = 3621.6 Å)
  Δλ at red end:     2.3838 Å  (at λ = 10353.8 Å)

  Spectral coverage: optical → near-IR
    Blue limit:      3622 Å  (362 nm) — near-UV/blue
    Red limit:       10354 Å (1035 nm) — near-IR (z/Y band)


In [34]:
# The WAVE extension gives the wavelength (in Å) for each slice index.
# To get the wavelength in nm for a specific slice:

with fits.open(fits_file) as hdul:
    wave_ang = hdul['WAVE'].data   # shape (4563,), in Angstroms
    wave_nm = wave_ang / 10.0      # convert to nm

    # Example: get wavelength of slice 0, 1000, 2500, 4562
    for i in [0, 1000, 2500, 4562]:
        print(f"  Slice {i:>4d}  →  {wave_nm[i]:.2f} nm  ({wave_ang[i]:.2f} Å)")

    print()

    # Reverse: find the closest slice for a given wavelength in nm
    target = 550.0  # nm (e.g. V-band center)
    idx = np.argmin(np.abs(wave_nm - target))
    print(f"  Closest slice to {target} nm  →  index {idx}  ({wave_nm[idx]:.2f} nm)")

  Slice    0  →  362.16 nm  (3621.60 Å)
  Slice 1000  →  455.93 nm  (4559.32 Å)
  Slice 2500  →  644.02 nm  (6440.21 Å)
  Slice 4562  →  1035.38 nm  (10353.81 Å)

  Closest slice to 550.0 nm  →  index 1815  (550.05 nm)
